# UserProxyAgent and Human Input

Not every turn should come from an LLM. **`UserProxyAgent`** represents the **human** in AgentChat: collecting console input, approving tool runs, or injecting reviewer feedback. This notebook covers **`input_func`**, **human-in-the-loop approval**, **`Console`** interactive loops, combining **user + assistant** in a **team**, **blocking vs async** human input, and comparison to **LangGraph HITL** (**02. LangGraph/09**).

Prerequisites: **03. AssistantAgent and ConversableAgent**. Next: **05. Two-Agent Conversation Patterns**.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import asyncio
import importlib.metadata as im
import os

os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

for pkg in ("autogen-agentchat", "autogen-ext", "autogen-core"):
    try:
        print(f"{pkg}:", im.version(pkg))
    except im.PackageNotFoundError:
        print(f"{pkg}: not installed — pip install autogen-agentchat autogen-ext[openai]")

In [ ]:
NOTES = {
    "n1": "The Notes API is built with FastAPI. Routes live under /notes.",
    "n2": "Run alembic upgrade head after pulling database migrations.",
    "n3": "JWT bearer tokens use Authorization: Bearer header.",
    "n4": "Pytest fixtures for DB setup belong in conftest.py.",
}

DOC_CHUNKS = [
    {"id": "c1", "text": NOTES["n1"]},
    {"id": "c2", "text": NOTES["n2"]},
    {"id": "c3", "text": NOTES["n3"]},
    {"id": "c4", "text": NOTES["n4"]},
    {"id": "c5", "text": "Use httpx.AsyncClient in FastAPI tests for async endpoints."},
]

print("Notes corpus:", list(NOTES.keys()))

---

## 1. Why UserProxyAgent?

v0.2 used **`UserProxyAgent`** for humans **and** code execution. In 0.4:

| Role | Agent |
|------|-------|
| Human input / approval | **`UserProxyAgent`** |
| LLM reasoning | **`AssistantAgent`** |
| Code execution | **`CodeExecutorAgent`** (**07**) |

```
task ──► AssistantAgent drafts ──► UserProxyAgent (human reviews) ──► APPROVE / feedback
              ▲                              │
              └──────── loop until APPROVE ──┘
```

---

## 2. UserProxyAgent Basics

| Parameter | Purpose |
|-----------|--------|
| **`name`** | Speaker name in transcripts (e.g. `"user"`) |
| **`input_func`** | Callable returning user text; defaults to built-in console input |
| **`description`** | Shown to other agents in handoff scenarios |

In [ ]:
from autogen_agentchat.agents import UserProxyAgent

default_user = UserProxyAgent(name="user")
print("default user agent:", default_user.name)

---

## 3. input_func Pattern

Customize how user text is collected — sync or async:

In [ ]:
APPROVAL_QUEUE: list[str] = []


def scripted_input(prompt: str = "") -> str:
    """Deterministic input for notebooks — no blocking stdin."""
    if APPROVAL_QUEUE:
        reply = APPROVAL_QUEUE.pop(0)
        print(f"[scripted user] {reply}")
        return reply
    return "APPROVE"


scripted_user = UserProxyAgent(name="reviewer", input_func=scripted_input)
print("scripted user ready")

For **async** servers, provide an async `input_func` that awaits a webhook, WebSocket message, or `asyncio.Queue` — never call blocking `input()` on the event loop.

---

## 4. Async input_func with asyncio.Queue

In [ ]:
user_queue: asyncio.Queue[str] = asyncio.Queue()


async def async_input(prompt: str = "") -> str:
    print("waiting for human via queue...")
    return await user_queue.get()


async_user = UserProxyAgent(name="async_reviewer", input_func=async_input)

await user_queue.put("APPROVE")
print("queued response:", await async_input())

---

## 5. Human-in-the-Loop Approval

Pair **`TextMentionTermination("APPROVE")`** with a critic or user proxy so runs end only after human sign-off:

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient

client = OpenAIChatCompletionClient(model="gpt-4o-mini", api_key=OPENAI_API_KEY)

writer = AssistantAgent(
    name="writer",
    model_client=client,
    system_message="Draft short answers about the Notes API. End drafts with 'AWAITING REVIEW'.",
)

APPROVAL_QUEUE[:] = ["Please mention note n1 for FastAPI routes.", "APPROVE"]
reviewer = UserProxyAgent(name="reviewer", input_func=scripted_input)

termination = TextMentionTermination("APPROVE")
review_team = RoundRobinGroupChat([writer, reviewer], termination_condition=termination)

---

## 6. Run the Approval Team with Console

In [ ]:
from autogen_agentchat.ui import Console


async def approval_demo() -> None:
    stream = review_team.run_stream(
        task="Explain where Notes API routes are defined."
    )
    await Console(stream)


await approval_demo()

The writer revises until the reviewer types **APPROVE** — same pattern as a critic agent, but the reviewer is human (or scripted for teaching).

---

## 7. Console Interactive Loop Sketch

Production consoles use **`input_func=input`** (built-in) for real stdin:

```python
live_user = UserProxyAgent("user", input_func=input)
team = RoundRobinGroupChat([assistant, live_user], termination_condition=...)
await Console(team.run_stream(task="..."))
```

In shared notebooks, prefer **scripted** or **queue** input to avoid blocking other readers.

---

## 8. Combining User + Assistant — Round Robin Flow

```
RoundRobinGroupChat([assistant, user_proxy])

  turn 1: assistant responds to task
  turn 2: user_proxy calls input_func → user text
  turn 3: assistant responds to user text
  ... until termination
```

Two-agent patterns without teams: **05. Two-Agent Conversation Patterns**.

---

## 9. Blocking vs Async Human Input

| Mode | Mechanism | Use when |
|------|-----------|----------|
| **Blocking** | `input()` in sync `input_func` | Local CLI demos only |
| **Scripted** | Pre-filled queue / list | Notebooks, unit tests |
| **Async queue** | `await queue.get()` | FastAPI + WebSocket HITL |
| **Webhook resume** | External ticket system | Enterprise approval chains |

LangGraph persists state at **`interrupt`** boundaries (**09**); AutoGen expects your **`input_func`** to bridge the wait.

---

## 10. MaxMessageTermination Safety Cap

Always cap turns when humans can loop indefinitely:

In [ ]:
from autogen_agentchat.conditions import MaxMessageTermination

safe_termination = TextMentionTermination("APPROVE") | MaxMessageTermination(8)
safe_team = RoundRobinGroupChat(
    [writer, reviewer],
    termination_condition=safe_termination,
)
print("termination uses APPROVE or max 8 messages")

Full guardrail catalog: **13. Termination Conditions and Guardrails**.

---

## 11. Comparison — AutoGen HITL vs LangGraph Interrupts

| | **AutoGen `UserProxyAgent`** | **LangGraph `interrupt`** (**09**) |
|--|------------------------------|-------------------------------------|
| **Model** | Agent turn in conversation | Graph pauses before/after node |
| **State** | Team message history | Checkpointed graph state |
| **Resume** | Next `input_func` call | `Command(resume=...)` with checkpointer |
| **Audit** | Message transcript | State snapshot + thread id |
| **Best for** | Chat-style review | Regulated step gates |

Use **LangGraph** when you must prove which node was blocked; use **AutoGen** when the human is just another conversational participant.

---

## 12. Notes API Review Scenario

Human verifies migration instructions before publishing:

In [ ]:
APPROVAL_QUEUE[:] = [
    "Add the exact alembic command from the docs.",
    "APPROVE",
]


async def migration_review() -> None:
    result = await safe_team.run(
        task="Write a one-line migration reminder for the Notes API team."
    )
    print("final:", result.messages[-1].content)
    print("stop_reason:", result.stop_reason)


await migration_review()

---

## 13. Handoffs to User (Preview)

Agents can **`handoff`** to a user agent when they need help — covered in **11. Sequential and Hierarchical Workflows**. Pattern:

```python
from autogen_agentchat.base import Handoff

assistant = AssistantAgent(
    ...,
    handoffs=[Handoff(target="user", message="Need approval")],
)
```

---

## 14. FastAPI Integration Sketch

```
Browser ──POST /approve──► FastAPI ──queue.put──► UserProxyAgent input_func
                                ▲
                                └── team.run_stream in background task
```

Store `asyncio.Queue` per session id; never block the worker with `input()`. Production patterns: **16. Production Patterns for AutoGen**.

---

## 15. v0.2 UserProxyAgent Mapping

| v0.2 | 0.4 AgentChat |
|------|---------------|
| `UserProxyAgent(name, human_input_mode="ALWAYS")` | `UserProxyAgent(name, input_func=...)` |
| `human_input_mode="NEVER"` | Remove user agent; use `AssistantAgent` only |
| `code_execution_config` on user proxy | `CodeExecutorAgent` in team (**07**) |
| `initiate_chat(assistant, ...)` | `RoundRobinGroupChat([user, assistant]).run(...)` |

In [ ]:
await client.close()
print("client closed")

---

## 16. Learning Path — This Series

```
01 Introduction ──► 02 Setup/AgentChat API ──► 03 AssistantAgent
     │
     ▼
04 UserProxy/HITL ──► 05 Two-agent patterns ──► 06 Tools
     │
     ▼
07 Code execution ──► 08 GroupChat teams ──► 09 Speaker selection
     │
     ▼
10 Custom roles ──► 11 Sequential/hierarchical ──► 12 Memory/state
     │
     ▼
13 Termination/guardrails ──► 14 AutoGen Studio ──► 15 Observability
     │
     ▼
16 Production patterns
```

---

## 17. Common Beginner Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| `input()` in FastAPI async route | Blocks event loop | Async `input_func` + queue |
| No `MaxMessageTermination` | Infinite human/LLM loop | Combine with `TextMentionTermination` |
| Expecting v0.2 code execution on user proxy | Wrong agent | Use `CodeExecutorAgent` (**07**) |
| Wrong team order | User speaks before context | Put assistant first or set clear task |
| Skipping LangGraph HITL comparison | Wrong framework choice | Read **02. LangGraph/09** |

---

## 18. Summary

`UserProxyAgent` brings **humans into the agent loop** via **`input_func`**, paired with **`AssistantAgent`** in **teams** and **`TextMentionTermination`** for approval. Use **scripted** or **async queue** input in notebooks and servers; reserve blocking **`input()`** for local CLI demos.

Key takeaways:

- v0.2 human + code proxy split: human → **`UserProxyAgent`**, code → **`CodeExecutorAgent`**.
- **`input_func`** is the extension point for WebSocket, queue, or scripted HITL.
- **`Console(team.run_stream(...))`** drives interactive multi-agent sessions.
- Combine with **`MaxMessageTermination`** to prevent runaway loops.
- LangGraph **interrupts** offer stronger checkpointed gates; AutoGen offers conversational HITL.

Next: **05. Two-Agent Conversation Patterns** — writer/critic, assistant/user, and termination without full group chat.